In [1]:
import pandas as pd

df = pd.read_csv("my_dataset.csv")

In [2]:
df.shape

(8173, 46)

In [3]:
df.head()

,id,event_type,latitude,longitude,endlatitude,endlongitude,address,end_address,event_cause,requires_road_closure,...,resolved_at_address,resolved_at_latitude,resolved_at_longitude,closed_by_id,closed_datetime,resolved_by_id,resolved_datetime,gba_identifier,zone,junction
0,FKID000000,unplanned,13.040004,77.518099,0.000000,0.000000,"Mumbai Bengaluru Highway, Jalahalli Cross Junc...",NaN,vehicle_breakdown,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,FKID000001,unplanned,12.921876,77.645158,0.000000,0.000000,"19th Main Road, Heavie Halcyon, Agara, HSR Lay...",NaN,vehicle_breakdown,False,...,"19th Main Road, Heavie Halcyon, Agara, HSR Lay...",12.921876,77.645158,NaN,NaN,FKUSR00002,2024-01-30 04:17:46.828355+00,NaN,NaN,NaN
2,FKID000002,unplanned,12.955622,77.585708,0.000000,0.000000,"Lalbagh Main Road, Dr Sri Shantaveera Swami Ci...",NaN,others,False,...,NaN,NaN,NaN,FKUSR00003,2024-01-30 04:56:03.281509+00,NaN,NaN,Bengaluru Central Corporation,Central Zone 2,UrvashiJunction
3,FKID000003,unplanned,13.006147,77.579435,13.006239,77.579516,"Sankey Road, Bashyam Circle, Sadashiva Nagar, ...","Sankey Road, Palace Orchard Upper, Sadashiva N...",tree_fall,True,...,NaN,NaN,NaN,FKUSR00004,2024-03-14 07:42:05.54944+00,NaN,NaN,NaN,NaN,NaN
4,FKID000004,unplanned,12.953980,77.585233,0.000000,0.000000,"Lalbagh Fort Road, Lalbagh Main Gate Junction,...",NaN,vehicle_breakdown,False,...,NaN,NaN,NaN,FKUSR00003,2024-01-30 05:35:17.338283+00,NaN,NaN,NaN,NaN,LalbaghMainGateJunc


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8173 entries, 0 to 8172
Data columns (total 46 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     8173 non-null   object 
 1   event_type             8173 non-null   object 
 2   latitude               8173 non-null   float64
 3   longitude              8173 non-null   float64
 4   endlatitude            8004 non-null   float64
 5   endlongitude           8004 non-null   float64
 6   address                8170 non-null   object 
 7   end_address            687 non-null    object 
 8   event_cause            8173 non-null   object 
 9   requires_road_closure  8173 non-null   bool   
 10  start_datetime         8173 non-null   object 
 11  end_datetime           490 non-null    object 
 12  status                 8173 non-null   object 
 13  authenticated          8173 non-null   object 
 14  modified_datetime      8173 non-null   object 
 15  map_

In [5]:
df["requires_road_closure"].value_counts()

,count
requires_road_closure,
False,7497
True,676


In [6]:
features = [
    "event_type",
    "event_cause",
    "veh_type",
    "corridor",
    "latitude",
    "longitude"
]

In [7]:
categorical_features = [
    "event_type",
    "event_cause",
    "veh_type",
    "corridor"
]

numerical_features = [
    "latitude",
    "longitude"
]

In [8]:
df = df[features + ["requires_road_closure"]]

In [9]:
df["veh_type"] = df["veh_type"].fillna("unknown")

In [10]:
X = df[features]

y = df["requires_road_closure"]

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    [
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            "passthrough",
            numerical_features
        )
    ]
)

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42
)
    )
])

In [14]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['event_type', 'event_cause',
                                                   'veh_type', 'corridor']),
                                                 ('num', 'passthrough',
                                                  ['latitude', 'longitude'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced',
                                        n_estimators=300, random_state=42))])

In [15]:
predictions = model.predict(X_test)

In [16]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, predictions)

0.9100917431192661

In [17]:
import joblib

joblib.dump(
    model,
    "road_closure_model.pkl"
)

['road_closure_model.pkl']

In [18]:
from google.colab import files

files.download(
    "road_closure_model.pkl"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**TESTING**


In [19]:
from sklearn.metrics import classification_report

print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

       False       0.94      0.97      0.95      1498
        True       0.44      0.29      0.35       137

    accuracy                           0.91      1635
   macro avg       0.69      0.63      0.65      1635
weighted avg       0.90      0.91      0.90      1635



In [20]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, predictions))

[[1448   50]
 [  97   40]]


In [21]:
from sklearn.metrics import roc_auc_score

probs = model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, probs)

print("ROC AUC:", roc_auc)

ROC AUC: 0.7577378109986064


In [22]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()

importances = model.named_steps["classifier"].feature_importances_

import pandas as pd

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
})

importance_df.sort_values(
    "importance",
    ascending=False
).head(20)

,feature,importance
52,num__latitude,0.294882
53,num__longitude,0.288978
14,cat__event_cause_tree_fall,0.038680
28,cat__veh_type_unknown,0.033055
0,cat__event_type_planned,0.028411
1,cat__event_type_unplanned,0.028034
15,cat__event_cause_vehicle_breakdown,0.025018
40,cat__corridor_Non-corridor,0.022359
6,cat__event_cause_construction,0.021418
8,cat__event_cause_pot_holes,0.014138


In [23]:
import joblib

joblib.dump(
    model,
    "road_closure_model_v2.pkl"
)

['road_closure_model_v2.pkl']